In [1]:
import sys
print(sys.executable)

d:\mlops-kredit-sumatera\.conda\python.exe


Eksplorasi 4 pertanyaan

In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/kredit_sumatera.csv")

# Pertanyaan 1: seberapa tidak seimbang targetnya?
print("Tingkat gagal bayar:", f"{df.gagal_bayar.mean():.2%}")

# Pertanyaan 2: apakah populasi berubah sepanjang waktu?
print(pd.crosstab(df.periode.str[:4], df.segmen, normalize="index").round(3))

# Pertanyaan 3: kolom mana yang punya nilai hilang, dan mengapa?
hilang = df.isna().mean().sort_values(ascending=False)
print(hilang[hilang > 0].round(4))
print(df.groupby("segmen")["rata_saldo_90h"].apply(lambda s: s.isna().mean()).round(3))

# Pertanyaan 4: adakah kolom yang mencurigakan?
print(df.groupby("gagal_bayar")["status_restrukturisasi"].apply(lambda s: s.notna().mean()))

Tingkat gagal bayar: 8.30%
segmen   nelayan  pedagang_musiman  pegawai_tetap  petani  wiraswasta
periode                                                              
2022       0.067             0.139          0.421    0.13       0.242
2023       0.067             0.142          0.419    0.13       0.241
2024       0.099             0.171          0.339    0.16       0.231
status_restrukturisasi    0.9479
lama_usaha_tahun          0.0294
rata_saldo_90h            0.0231
dtype: float64
segmen
nelayan             0.108
pedagang_musiman    0.000
pegawai_tetap       0.000
petani              0.105
wiraswasta          0.000
Name: rata_saldo_90h, dtype: float64
gagal_bayar
0    0.000000
1    0.627509
Name: status_restrukturisasi, dtype: float64


1. Tingkat gagal bayar 8,30% → persis sesuai modul. Data sangat tidak seimbang: dari 100 nasabah, hanya sekitar 8 yang gagal bayar. Karena itu akurasi tidak cocok dipakai (menebak “semua lancar” saja sudah 91,7% benar), makanya modul memakai AUC.

2. Populasi bergeser di 2024. Pegawai tetap turun dari 42,1% (2022) ke 33,9% (2024), sedangkan nelayan naik dari 6,7% ke 9,9%, petani 13% ke 16%, dan pedagang musiman 13,9% ke 17,1%. Segmen yang naik justru segmen berisiko tinggi, jadi model yang dilatih dengan data 2022–2023 akan melayani populasi yang berbeda (bahan Praktikum 2).

3. Nilai hilang tidak acak. rata_saldo_90h hanya hilang di segmen nelayan (10,8%) dan petani (10,5%), segmen lain 0%. Artinya kalau nilai hilang diisi median keseluruhan, kedua segmen itu diperlakukan keliru (titik periksa nomor 4).

4. status_restrukturisasi adalah kebocoran target. Kolom ini 0% terisi pada nasabah yang lancar dan 62,8% terisi pada nasabah gagal bayar. Jadi begitu kolom ini terisi, nasabah itu pasti gagal bayar. Masalahnya, kolom ini baru terisi setelah nasabah gagal bayar, sehingga tidak tersedia saat keputusan kredit diambil. Kalau dipakai sebagai fitur, model terlihat sangat bagus di data latih tapi tidak berguna di produksi (titik periksa nomor 5). Kolom ini juga hilang 94,8%, karena memang hanya terisi untuk sebagian kecil nasabah.

### Model garis dasar, pembagian menurut waktu

In [3]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df["rasio_angsuran"] = df.angsuran_bulanan / df.penghasilan_bulanan
num = ["usia", "tanggungan", "lama_usaha_tahun", "penghasilan_bulanan",
       "ragam_penghasilan", "plafon_diajukan", "tenor_bulan", "bunga_tahunan",
       "rasio_angsuran", "rata_saldo_90h", "jumlah_tunggakan_12b"]
X = df[num].join(pd.get_dummies(df[["segmen", "wilayah", "kanal"]]))
y = df.gagal_bayar

# Pembagian MENURUT WAKTU, bukan acak. Data ini bergantung waktu.
latih = df.periode <= "2023-12"
uji = df.periode > "2024-06"

m = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.06,
                                   max_iter=250, random_state=42)
m.fit(X[latih], y[latih])
skor = m.predict_proba(X[uji])[:, 1]
print("AUC out-of-time:", round(roc_auc_score(y[uji], skor), 4))

AUC out-of-time: 0.7744


Sesuai acuan modul (sekitar 0,775). Model dilatih dengan data sampai Desember 2023 dan diuji pada Juli–Desember 2024, persis seperti kondisi nyata: model selalu dipakai untuk memprediksi masa depan.

### Pembanding pembagian acak

In [4]:
from sklearn.model_selection import train_test_split

Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
m2 = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.06,
                                    max_iter=250, random_state=42).fit(Xa, ya)
print("AUC pembagian acak:", round(roc_auc_score(yb, m2.predict_proba(Xb)[:, 1]), 4))

AUC pembagian acak: 0.7631


Ini berbeda dari modul. Modul memperkirakan AUC acak lebih tinggi (sekitar 0,79), tapi hasil kamu justru lebih rendah dari out-of-time. Jangan diubah atau ditutupi, karena modul sendiri bilang perbedaan angka adalah sinyal untuk ditelusuri.

Kemungkinan penyebabnya versi library yang berbeda (scikit-learn 1.7.2 milik kita vs 1.8.0 milik dosen), ditambah satu kali pembagian acak bisa kebetulan menghasilkan data uji yang sulit. Untuk mengeceknya, tambahkan cell ini:

In [5]:
hasil = []
for seed in range(5):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    m_acak = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.06,
                                            max_iter=250, random_state=42).fit(Xa, ya)
    auc = roc_auc_score(yb, m_acak.predict_proba(Xb)[:, 1])
    hasil.append(auc)
    print(f"seed {seed}: AUC acak = {auc:.4f}")
print(f"Rata-rata AUC acak: {sum(hasil)/len(hasil):.4f}")

seed 0: AUC acak = 0.7771
seed 1: AUC acak = 0.7994
seed 2: AUC acak = 0.7785
seed 3: AUC acak = 0.7749
seed 4: AUC acak = 0.7798
Rata-rata AUC acak: 0.7820
